# 02. 평가 실행 (공통)

드라이브의 촬영 데이터로 파이프라인을 돌려 지표를 계산한다. 지표 정의는 `docs/eval_protocol.md`.

데이터 구조 (`docs/data_guide.md`)
```
DL-Team05/data/images/P001_back_normal_01.jpg ...
DL-Team05/data/labels/products.csv
DL-Team05/data/labels/boxes/*.json   (테스트셋만)
```

In [ ]:
# 레포 받기 (이미 있으면 최신으로) + 기본 설치
import os
REPO = "/content/AI_tts_web"
if not os.path.exists(REPO):
    !git clone -q https://github.com/DL-2026F-Team05/AI_tts_web.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r requirements.txt -r requirements-dev.txt
!git log --oneline -3

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/DL-Team05/data"
OUTS = "/content/drive/MyDrive/DL-Team05/outputs"

## 1. 데이터 규칙 검사

In [ ]:
!python -m tools.dataset --root "{DATA}"

## 2. 제품 단위 분할 (처음 한 번만)
만든 `data/splits/*.txt` 는 커밋해서 모두 같은 분할을 쓴다. 이미 있으면 건너뛴다.

In [ ]:
import os
if not os.path.exists("data/splits/test.txt"):
    !python -m tools.make_splits --root "{DATA}" --by-category
else:
    print("분할이 이미 있다:", open("data/splits/test.txt").read().split())

## 3. 평가
OCR 엔진을 바꿔 가며 같은 셀을 돌리면 비교 표가 생긴다.

In [ ]:
!pip install -q easyocr

In [ ]:
ENGINE = "easyocr"   # easyocr | paddle
!python -m evaluation.run_eval --root "{DATA}" --ocr {ENGINE} --out "{OUTS}/eval_{ENGINE}"

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{OUTS}/eval_{ENGINE}/summary.md", encoding="utf-8").read()))

## 4. 판단 보류 기준값 정하기 (coverage-risk)
기준값을 올리면 답하는 비율은 줄고, 답한 것 중 틀리는 비율도 준다.

In [ ]:
import json
import matplotlib.pyplot as plt
rep = json.load(open(f"{OUTS}/eval_{ENGINE}/report.json", encoding="utf-8"))
cr = rep.get("expiry", {}).get("coverage_risk", [])
if cr:
    plt.plot([r["coverage"] for r in cr], [r["error"] for r in cr], marker="o")
    for r in cr[::4]:
        plt.annotate(str(r["threshold"]), (r["coverage"], r["error"]))
    plt.xlabel("coverage (answered)")
    plt.ylabel("error among answered")
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("날짜가 보이는 사진이 없다")

## 5. 틀린 사례 보기

In [ ]:
import pandas as pd
df = pd.read_csv(f"{OUTS}/eval_{ENGINE}/per_image.csv")
cols = [c for c in ["image", "condition", "gt_date", "pred_date", "pred_status", "pred_conf", "cer"] if c in df]
wrong = df[(df.get("date_visible") == True) & (df.get("date_correct") == False)] if "date_correct" in df else df.head(0)
wrong[cols].head(20)